# PINN — Inverse Problem, Robustness & Diagnostics
## Complete Notebook (run top to bottom)

### What this notebook does, in order:

| Section | Content | Fixes / Adds |
|---------|---------|-------------|
| 1–3 | Setup, config, utilities | — |
| 4 | True bathymetry & bottom profile | — |
| 5 | Network & training definitions | — |
| 6 | **Forward PINN** (true d(x) fixed) | Generates non-circular observations |
| 7 | **Inverse PINN** — prototype (K=6, σ=0.001) | Verify before full run |
| 8 | **Ensemble UQ** (5 seeds) | Mean ± std of recovered d(x) |
| 9 | **Loss weight ablation** (w_IC × w_data) | Justifies weight choices |
| 10 | **Convergence analysis** across seeds | Reproducibility evidence |
| 11 | **Full ablation** (K × σ, PINN-generated obs) | Main results table |
| 12 | **κ₁ sensitivity** | How RMSE degrades with wrong κ₁ |
| 13 | **PDE residual fields** | Spatial error diagnosis |
| 14 | **Computational cost table** | Paper Table |
| 15 | Summary & all outputs |

## 0. Install Dependencies

In [ ]:
# !pip install tensorflow>=2.12.0 numpy scipy matplotlib

## 1. Imports & Reproducibility

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.interpolate import CubicSpline
import time, os

np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {len(tf.config.list_physical_devices("GPU")) > 0}')

SAVE_DIR = 'pinn_full_results'
os.makedirs(SAVE_DIR, exist_ok=True)

## 2. Configuration ← Adjust Here

In [ ]:
# ---------------------------------------------------------------
# Physical (Carrier & Greenspan geometry)
# ---------------------------------------------------------------
G, H0             = 9.81, 0.5
X_MIN, X_MAX      = -50.0, 10.0
T_MIN, T_MAX      = 0.0, 30.0
SLOPE_START       = -5.5
SLOPE             = 1.0 / 25.0
WAVE_AMP          = 0.03
WAVE_PERIOD       = 10.0

# Still-water shoreline: d(x) = 0 at x = SLOPE_START + H0/SLOPE
X_SHORE           = SLOPE_START + H0 / SLOPE          # = 7.0 m
H_MIN             = 1e-3 * H0                          # thin-film floor

# ---------------------------------------------------------------
# VBM closure.  α, β, γ are NOT free constants — they are the vertical
# moments of the profile F(z), fixed by κ₁d.  See §3 for the formulas and
# the dispersion-relation check that validates them.
#
# The previous hardcoded values (ALPHA_OPT=0.1169, BETA_OPT=0.1312) are
# inconsistent with those moments: they overpredict ω² at k=κ₁ by 42%,
# and β>0 is impossible (β = tanh(κd)/κ − d < 0 for all κd).
# ---------------------------------------------------------------
KAPPA1_D = 1.52       # optimal κ₁·d, held constant across the domain

# ---------------------------------------------------------------
# Ablation grids
# ---------------------------------------------------------------
GAUGE_COUNTS  = [3, 6, 10]           # K: number of gauges
NOISE_LEVELS  = [0.0, 0.001, 0.005]  # σ: observation noise [m]
N_ENSEMBLE    = 5                    # seeds for UQ

# Gauge placement — must extend onto the sloping bed, otherwise every
# observation sits at depth H0 and carries no information about the slope.
GAUGE_X_MIN   = X_MIN + 5.0          # -45.0 m (flat region)
GAUGE_X_MAX   = 5.0                  #   5.0 m (on the slope, d ≈ 0.08 m)

# Loss weight ablation
W_IC_GRID   = [5.0, 10.0, 20.0]
W_DATA_GRID = [10.0, 20.0, 40.0]

# κ₁ sensitivity
KAPPA1_D_VALUES = [1.0, 1.25, 1.52, 1.75, 2.0, 2.5]

# ---------------------------------------------------------------
# Bathymetry parameterization
# ---------------------------------------------------------------
POLY_DEGREE = 4    # Chebyshev polynomial degree for d(x)

# ---------------------------------------------------------------
# Network
# ---------------------------------------------------------------
LAYERS, ACTIVATION = [2, 64, 64, 64, 64, 3], 'tanh'

# ---------------------------------------------------------------
# Collocation
# ---------------------------------------------------------------
N_F, N_IC, N_BC = 8_000, 400, 200

# ---------------------------------------------------------------
# Training
# ---------------------------------------------------------------
LR            = 1e-3
EPOCHS_ADAM   = 6_000
EPOCHS_LBFGS  = 2_000
PLOT_FREQ     = 2_000

# ---------------------------------------------------------------
# Default loss weights
# ---------------------------------------------------------------
W_PDE  = 1.0
W_ELL  = 1.0
W_IC   = 10.0   # default; varied in ablation
W_BC   = 5.0
W_DATA = 20.0   # default; varied in ablation

n_ablation = len(GAUGE_COUNTS) * len(NOISE_LEVELS)
n_weight   = len(W_IC_GRID) * len(W_DATA_GRID)
n_kappa    = len(KAPPA1_D_VALUES)
print(f'Shoreline at x = {X_SHORE:.1f} m  (gauges span {GAUGE_X_MIN} … {GAUGE_X_MAX} m)')
print(f'Inverse ablation : {n_ablation} runs  ({len(GAUGE_COUNTS)} K × {len(NOISE_LEVELS)} σ)')
print(f'Weight ablation  : {n_weight} runs   ({len(W_IC_GRID)} w_IC × {len(W_DATA_GRID)} w_data)')
print(f'κ₁ sensitivity   : {n_kappa} runs')
print(f'Ensemble UQ      : {N_ENSEMBLE} seeds')
print(f'Total training runs ≈ {N_ENSEMBLE + n_ablation + n_weight + n_kappa + 2}')

## 3. Shared Utilities

In [ ]:
# ---- True bathymetry (NumPy).  Unclamped: negative above the shoreline. ----
def true_depth(x_arr):
    x = np.asarray(x_arr).flatten()
    d = np.where(x < SLOPE_START, H0, H0 - SLOPE*(x - SLOPE_START))
    return d.reshape(x_arr.shape) if hasattr(x_arr,'shape') else d


# ---- True bathymetry (TensorFlow).  Pure TF ⇒ differentiable in x. ----
def true_depth_tf(x):
    """
    d(x) as a differentiable TF op.

    The previous implementation wrapped true_depth in tf.py_function, which has
    no registered gradient.  ∂ₓ(hu) then silently lost the shoaling term u·∂ₓd —
    no error, just missing physics.  tf.where keeps the tape intact.
    """
    x0 = tf.constant(SLOPE_START, tf.float32)
    s  = tf.constant(SLOPE,       tf.float32)
    return tf.where(x < x0, H0 * tf.ones_like(x), H0 - s * (x - x0))


def smooth_pos(s, eps=H_MIN):
    """Smooth positive part: → s for s ≫ ε, → 0⁺ for s ≪ −ε.  C^∞ everywhere."""
    e = tf.constant(eps, tf.float32)
    return 0.5 * (s + tf.sqrt(s * s + e * e))


# ---- VBM coefficients ----
def vbm_shape(kappa1_d):
    """
    Scale factors (A, B, Gc) such that, with κ₁·d held fixed at kappa1_d,
        α = A·d,   β = B·d,   γ = Gc/d.

    Derived from the vertical moments of F(z) = cosh(κ(z+d))/cosh(κd) − 1:
        β = ∫F dz    = tanh(κd)/κ − d
        α = ∫F² dz   = d(1 + ½sech²κd − (3/2)tanh(κd)/(κd))
        γ = ∫(F')²dz = (κ/2)(tanh κd − κd sech²κd)
    """
    C  = float(kappa1_d)
    T  = np.tanh(C)
    S2 = 1.0 / np.cosh(C)**2
    return (1.0 + 0.5*S2 - 1.5*T/C,      # A
            T/C - 1.0,                    # B  (always negative)
            0.5*C*(T - C*S2))             # Gc


def vbm_coeffs(kappa1_d, d=H0):
    """α, β, γ at depth d."""
    A, B, Gc = vbm_shape(kappa1_d)
    return A*d, B*d, Gc/d


def coeffs_tf(d_pos, kappa1_d):
    """α(x), β(x), γ(x) from a positive local depth tensor."""
    A, B, Gc = vbm_shape(kappa1_d)
    return (tf.constant(A,  tf.float32) * d_pos,
            tf.constant(B,  tf.float32) * d_pos,
            tf.constant(Gc, tf.float32) / d_pos)


# ---- Dispersion relation: what actually pins α, β, γ down ----
def omega2_vbm(k, d, kappa1_d):
    """Linearised VBM: ω² = g k² [ d − β²k²/(αk² + γ) ]."""
    a, b, g_ = vbm_coeffs(kappa1_d, d)
    return G * k**2 * (d - b**2 * k**2 / (a * k**2 + g_))


def omega2_exact(k, d):
    return G * k * np.tanh(k * d)


# ---- Metrics ----
def bathy_metrics(d_true, d_pred):
    rmse = np.sqrt(np.mean((d_true - d_pred)**2))
    corr = np.dot(d_true, d_pred) / (np.linalg.norm(d_true)*np.linalg.norm(d_pred) + 1e-12)
    return float(rmse), float(corr)


# ---- Collocation sampling ----
def sample_collocation(seed=0):
    rng = np.random.default_rng(seed)
    return {
        'x_f' : tf.constant(rng.uniform(X_MIN, X_MAX, (N_F, 1)).astype(np.float32)),
        't_f' : tf.constant(rng.uniform(T_MIN, T_MAX, (N_F, 1)).astype(np.float32)),
        'x_ic': tf.constant(rng.uniform(X_MIN, X_MAX, (N_IC,1)).astype(np.float32)),
        't_ic': tf.constant(np.zeros((N_IC,1), dtype=np.float32)),
        'x_bc': tf.constant(np.full((N_BC,1), X_MIN, dtype=np.float32)),
        't_bc': tf.constant(rng.uniform(T_MIN, T_MAX, (N_BC,1)).astype(np.float32)),
    }


# ---- Observation tensor builder ----
def build_obs(gauge_x, obs_t, obs_eta):
    K, Nt = obs_eta.shape
    return {
        'x'  : tf.constant(np.repeat(gauge_x, Nt).reshape(-1,1).astype(np.float32)),
        't'  : tf.constant(np.tile(obs_t, K).reshape(-1,1).astype(np.float32)),
        'eta': tf.constant(obs_eta.flatten().reshape(-1,1).astype(np.float32)),
    }


# ---- Spline baseline ----
# NOTE: this baseline is handed the TRUE depth at each gauge, while the PINN
# only ever sees η.  It is an unequal-information reference and must be
# described as such wherever the comparison is reported.
def spline_baseline(gauge_x, obs_depth, x_eval):
    xa = np.concatenate([[X_MIN], gauge_x, [X_MAX]])
    da = np.concatenate([[H0],    obs_depth, [true_depth(np.array([X_MAX]))[0]]])
    return CubicSpline(xa, da, bc_type='not-a-knot')(x_eval)


# ---- Evaluation grid ----
x_eval      = np.linspace(X_MIN, X_MAX, 400)
d_true_eval = true_depth(x_eval)

print('Utilities defined ✓')

# ---- Verify the coefficients reproduce exact dispersion at k = κ₁ ----
print('\nVBM coefficients (d = %.2f m) and dispersion check at k = κ₁:' % H0)
print(f'{"κ₁d":>6}  {"α":>9}  {"β":>9}  {"γ":>8}  {"ω²(VBM)":>10}  {"ω²(exact)":>10}  {"err":>8}')
print('-'*72)
for k1d in KAPPA1_D_VALUES:
    a, b, g_ = vbm_coeffs(k1d)
    k        = k1d / H0
    w_v, w_e = omega2_vbm(k, H0, k1d), omega2_exact(k, H0)
    marker   = ' ← optimal' if k1d == KAPPA1_D else ''
    print(f'{k1d:>6.2f}  {a:>9.4f}  {b:>9.4f}  {g_:>8.4f}  '
          f'{w_v:>10.4f}  {w_e:>10.4f}  {abs(w_v/w_e-1):>8.1e}{marker}')

_k   = KAPPA1_D / H0
_err = abs(omega2_vbm(_k, H0, KAPPA1_D) / omega2_exact(_k, H0) - 1.0)
assert _err < 1e-3, 'VBM coefficients do not reproduce exact dispersion at k=κ₁'
print(f'\n✓ coefficients verified (rel. error {_err:.1e} at κ₁d={KAPPA1_D})')

# For reference: what the old hardcoded constants implied
_a_old, _b_old, _g_old = 0.1169, 0.1312, 1.0
_w_old = G*_k**2*(H0 - _b_old**2*_k**2/(_a_old*_k**2 + _g_old))
print(f'  (old hardcoded α=0.1169 β=0.1312 γ=1.0 → ω²={_w_old:.4f}, '
      f'{abs(_w_old/omega2_exact(_k,H0)-1)*100:.0f}% error — discarded)')

# ---- Verify the TF depth is differentiable and matches NumPy ----
_xc = tf.constant(x_eval.astype(np.float32)[:, None])
with tf.GradientTape() as _tp:
    _tp.watch(_xc)
    _dc = true_depth_tf(_xc)
_grad = _tp.gradient(_dc, _xc)
assert _grad is not None, 'true_depth_tf is not differentiable — check for py_function'
_gm = _grad.numpy().flatten()
print(f'\ntrue_depth_tf: max |Δd| vs NumPy = {np.abs(_dc.numpy().flatten()-d_true_eval).max():.2e}')
print(f'               ∂ₓd = {_gm.min():.4f} (slope) … {_gm.max():.4f} (flat)  '
      f'— expected −{SLOPE:.4f} and 0.0  ✓')

## 4. True Bathymetry Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10,3))
ax.fill_between(x_eval, -d_true_eval, -d_true_eval.max()-0.05,
                color='wheat', alpha=0.7, label='True bed  −d(x)')
ax.axhline(0, color='steelblue', lw=1.5, label='Still water')
ax.axvline(SLOPE_START, color='gray', ls=':',  lw=1.2, label=f'Slope toe (x={SLOPE_START})')
ax.axvline(X_SHORE,     color='crimson', ls='--', lw=1.2, label=f'Shoreline (x={X_SHORE:.1f})')
ax.set(title='True Bathymetry — Carrier & Greenspan Geometry',
       xlabel='x [m]', ylabel='z [m]')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'true_bathymetry.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'd(x) range over the domain: [{d_true_eval.min():.3f}, {d_true_eval.max():.3f}] m')
print(f'Dry at rest beyond x = {X_SHORE:.1f} m '
      f'({100*np.mean(d_true_eval <= 0):.0f}% of the domain) — '
      f'handled by the thin-film regularisation, not by clamping d.')

## 5. Network & Training Definitions

In [ ]:
# ============================================================
#  Solution network: (x,t) → (η, u, Ψ)
# ============================================================
class SolutionNet(tf.keras.Model):
    def __init__(self, layers_cfg=LAYERS, act=ACTIVATION):
        super().__init__()
        self.lb = tf.constant([[X_MIN, T_MIN]], dtype=tf.float32)
        self.ub = tf.constant([[X_MAX, T_MAX]], dtype=tf.float32)
        self.hidden = [
            tf.keras.layers.Dense(u, activation=act, kernel_initializer='glorot_normal')
            for u in layers_cfg[1:-1]
        ]
        self.out = tf.keras.layers.Dense(layers_cfg[-1], activation=None,
                                          kernel_initializer='glorot_normal')
    def call(self, x, t):
        xt = tf.concat([x, t], axis=1)
        h  = 2.*(xt - self.lb)/(self.ub - self.lb) - 1.
        for l in self.hidden: h = l(h)
        o = self.out(h)
        return o[:,0:1], o[:,1:2], o[:,2:3]


# ============================================================
#  Bathymetry model: d(x) as a Chebyshev polynomial
# ============================================================
class BathyNet(tf.Module):
    """
    d(x) = Σ c_k T_k(ξ(x))    — raw Chebyshev expansion, no positivity transform.

    The softplus wrapper was dropped: the true bed rises above still water at
    x > X_SHORE, so d(x) must be free to go negative.  Positivity is enforced
    only where it is actually required (the α, β, γ formulas and the water
    column h), via smooth_pos in compute_residuals.

    c[0] = d0 and all higher modes zero ⇒ d(x) ≡ d0 at initialisation.
    """
    def __init__(self, deg=POLY_DEGREE, d0=H0):
        super().__init__()
        self.deg = deg
        self.lb  = tf.constant(X_MIN, dtype=tf.float32)
        self.ub  = tf.constant(X_MAX, dtype=tf.float32)
        init     = np.zeros(deg+1, dtype=np.float32)
        init[0]  = d0
        self.c   = tf.Variable(init, trainable=True, name='bathy_coeffs')
    def _xi(self, x): return 2.*(x - self.lb)/(self.ub - self.lb) - 1.
    def _T(self, xi):
        T = [tf.ones_like(xi), xi]
        for k in range(2, self.deg+1): T.append(2.*xi*T[-1] - T[-2])
        return tf.concat(T[:self.deg+1], axis=1)
    def __call__(self, x): return self._T(self._xi(x)) @ tf.expand_dims(self.c, 1)
    def numpy(self, x_arr): return self(tf.constant(x_arr.reshape(-1,1), dtype=tf.float32)).numpy().flatten()


# ============================================================
#  PDE residuals — divergence form, coefficients follow the local depth
#
#    r_η = ∂ₜη + ∂ₓ(hu) + ∂ₓ(β ∂ₓΨ)
#    r_u = ∂ₜu + g∂ₓη + u∂ₓu
#    r_Ψ = −∂ₓ(α ∂ₓΨ) + γΨ − ∂ₓ(βu)
#
#  bathy_fn may be true_depth_tf (fixed) or a BathyNet (trainable); either way
#  it is differentiable in x, so ∂ₓ(hu) carries the shoaling term u·∂ₓd.
# ============================================================
def compute_residuals(sol, bathy_fn, x, t, kappa1_d=KAPPA1_D):
    with tf.GradientTape(persistent=True) as t2:
        t2.watch([x, t])
        with tf.GradientTape(persistent=True) as t1:
            t1.watch([x, t])
            eta, u, psi = sol(x, t)

            d     = bathy_fn(x)
            d_pos = smooth_pos(d) + H_MIN
            alpha, beta, gamma = coeffs_tf(d_pos, kappa1_d)

            h  = smooth_pos(d + eta)
            hu = h * u

        eta_t=t1.gradient(eta,t); eta_x=t1.gradient(eta,x)
        u_t  =t1.gradient(u,  t); u_x  =t1.gradient(u,  x)
        psi_x=t1.gradient(psi,x); hu_x =t1.gradient(hu, x)

        flux_b_psi = beta  * psi_x
        flux_a_psi = alpha * psi_x
        flux_b_u   = beta  * u

    d_flux_b_psi = t2.gradient(flux_b_psi, x)
    d_flux_a_psi = t2.gradient(flux_a_psi, x)
    d_flux_b_u   = t2.gradient(flux_b_u,   x)

    r_eta = eta_t + hu_x + d_flux_b_psi
    r_u   = u_t + G*eta_x + u*u_x
    r_psi = -d_flux_a_psi + gamma*psi - d_flux_b_u
    return r_eta, r_u, r_psi


# ============================================================
#  Total loss
# ============================================================
def compute_loss(sol, bathy_fn, coll, obs=None, kappa1_d=KAPPA1_D,
                 w_ic=W_IC, w_bc=W_BC, w_data=W_DATA):
    r1,r2,r3 = compute_residuals(sol, bathy_fn, coll['x_f'], coll['t_f'], kappa1_d)
    lp = tf.reduce_mean(tf.square(r1)) + tf.reduce_mean(tf.square(r2))
    le = tf.reduce_mean(tf.square(r3))
    ei,ui,_ = sol(coll['x_ic'], coll['t_ic'])
    li = tf.reduce_mean(tf.square(ei)) + tf.reduce_mean(tf.square(ui))
    amp=tf.constant(WAVE_AMP, dtype=tf.float32)
    Tw =tf.constant(WAVE_PERIOD, dtype=tf.float32)
    eb,_,_ = sol(coll['x_bc'], coll['t_bc'])
    lb = tf.reduce_mean(tf.square(eb - amp*tf.cos(2.*np.pi/Tw*coll['t_bc'])))
    ld = tf.constant(0.0)
    if obs is not None:
        ep,_,_ = sol(obs['x'], obs['t'])
        ld = tf.reduce_mean(tf.square(ep - obs['eta']))
    total = W_PDE*lp + W_ELL*le + w_ic*li + w_bc*lb + w_data*ld
    return total, lp, le, li, lb, ld


# ============================================================
#  Training: Adam + L-BFGS-B
# ============================================================
def train(sol, bathy_fn, coll, obs=None, kappa1_d=KAPPA1_D,
          w_ic=W_IC, w_bc=W_BC, w_data=W_DATA,
          epochs_adam=EPOCHS_ADAM, epochs_lbfgs=EPOCHS_LBFGS,
          verbose=False):
    """Joint optimization of sol + bathy_fn (if BathyNet)."""
    extra = list(bathy_fn.trainable_variables) if isinstance(bathy_fn, BathyNet) else []
    all_v = sol.trainable_variables + extra
    opt   = tf.keras.optimizers.Adam(learning_rate=LR)
    hist  = {k:[] for k in ['total','pde','ell','ic','bc','data']}

    @tf.function
    def step():
        with tf.GradientTape() as tape:
            vals = compute_loss(sol, bathy_fn, coll, obs, kappa1_d, w_ic, w_bc, w_data)
        g = tape.gradient(vals[0], all_v)
        opt.apply_gradients(zip(g, all_v))
        return vals

    t0 = time.time()
    for ep in range(1, epochs_adam+1):
        vals = step()
        for k,v in zip(hist.keys(), vals): hist[k].append(float(v))
        if verbose and (ep % PLOT_FREQ == 0 or ep == 1):
            print(f'    Adam {ep:>6}/{epochs_adam} | total={float(vals[0]):.3e} '
                  f'data={float(vals[5]):.3e} | {time.time()-t0:.1f}s')
    t_adam = time.time() - t0

    n_sol = len(sol.trainable_variables)
    def lbfgs_fn(w):
        idx, nw = 0, []
        for v in all_v:
            sz = tf.size(v).numpy(); nw.append(w[idx:idx+sz].reshape(v.shape)); idx += sz
        sol.set_weights(nw[:n_sol])
        for v,ww in zip(extra, nw[n_sol:]): v.assign(ww)
        with tf.GradientTape() as tape:
            vals = compute_loss(sol, bathy_fn, coll, obs, kappa1_d, w_ic, w_bc, w_data)
        g = tape.gradient(vals[0], all_v)
        for k,v in zip(hist.keys(), vals): hist[k].append(float(v))
        return float(vals[0]), np.concatenate([gg.numpy().flatten() for gg in g]).astype(np.float64)

    t1  = time.time()
    w0  = np.concatenate([v.numpy().flatten() for v in all_v]).astype(np.float64)
    res = minimize(lbfgs_fn, w0, method='L-BFGS-B', jac=True,
                   options={'maxiter':epochs_lbfgs,'ftol':1e-12,'gtol':1e-8})
    t_lbfgs = time.time() - t1
    if verbose: print(f'    LBFGS: {res.message} | loss={res.fun:.3e}')
    return hist, t_adam, t_lbfgs


print('Networks, residuals, and training function defined ✓')

# ---- sanity: BathyNet starts flat at H0, and the residual sees ∂ₓd ----
_bm = BathyNet()
print(f'BathyNet at init: d range over domain = '
      f'[{_bm.numpy(x_eval).min():.4f}, {_bm.numpy(x_eval).max():.4f}] m  (expect {H0})')

_s  = SolutionNet(); _c = sample_collocation(seed=0)
_ = _s(_c['x_ic'], _c['t_ic'])
for _name, _bf in [('true_depth_tf', true_depth_tf), ('BathyNet', _bm)]:
    _r = compute_residuals(_s, _bf, _c['x_f'][:256], _c['t_f'][:256])
    assert all(np.all(np.isfinite(v.numpy())) for v in _r), f'non-finite residual ({_name})'
    print(f'  {_name:<14} |r_η|={np.abs(_r[0].numpy()).mean():.3e}  '
          f'|r_u|={np.abs(_r[1].numpy()).mean():.3e}  |r_Ψ|={np.abs(_r[2].numpy()).mean():.3e}')
print(f'Params per solution net: {_s.count_params():,}')

## 6. Forward PINN with True Bathymetry

Train a forward PINN with **true d(x) fixed** (not trainable). Its η predictions become the
synthetic observations for the inverse problem.

> **Why this matters:** if the observations came from the same closed-form expression the
> inverse model is fitting, the recovery would be trivially easy. Drawing them from a
> converged forward network removes that shortcut.
>
> **What it does not test:** the forward and inverse models share the same PDE family and the
> same κ₁d, so this measures **identifiability of d(x)** — whether the gauge data constrain the
> bed — not robustness to model-form error. A genuinely independent source (a finite-volume
> solver, or laboratory data) would be needed for the latter.

`true_depth_tf` is now a pure-TF op, so ∂ₓd reaches the tape and the forward solution actually
shoals. Under the previous `tf.py_function` wrapper it did not.

In [ ]:
# true_depth_tf is defined in §3 as a pure-TF op (differentiable in x).
# It replaces the earlier tf.py_function wrapper, which had no gradient and
# silently dropped u·∂ₓd from ∂ₓ(hu).

print('Training forward PINN (true bathymetry, fixed)...')
np.random.seed(0); tf.random.set_seed(0)
fwd_sol  = SolutionNet()
fwd_coll = sample_collocation(seed=0)
_ = fwd_sol(fwd_coll['x_ic'], fwd_coll['t_ic'])
fwd_sol.summary()

t_fwd_start = time.time()
hist_fwd, t_adam_fwd, t_lbfgs_fwd = train(
    fwd_sol, true_depth_tf, fwd_coll, obs=None, verbose=True
)
t_fwd_total = time.time() - t_fwd_start
print(f'\nForward PINN trained in {t_fwd_total:.1f}s '
      f'(Adam: {t_adam_fwd:.1f}s  LBFGS: {t_lbfgs_fwd:.1f}s)')

# Plot forward loss
ep_fwd = np.arange(1, len(hist_fwd['total'])+1)
fig, ax = plt.subplots(figsize=(9,3))
ax.semilogy(ep_fwd, hist_fwd['total'], 'k', lw=1.5, label='Total')
for k,col in [('pde','steelblue'),('ell','darkorange'),('ic','green'),('bc','red')]:
    ax.semilogy(ep_fwd, hist_fwd[k], color=col, lw=1, label=k)
ax.axvline(EPOCHS_ADAM, color='gray', ls='--', label='Adam→LBFGS')
ax.set(title='Forward PINN Training Loss', xlabel='Epoch', ylabel='Loss')
ax.legend(ncol=3, fontsize=8); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'fwd_loss.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def gauge_positions(n_gauges):
    """
    Gauge layout for K sensors.

    The old layout was linspace(X_MIN+5, SLOPE_START-2) — entirely inside the
    flat region, so every gauge sat at depth H0 and no observation carried any
    information about the slope.

    A plain linspace over the full span is barely better: the slope is only
    12.5 m of a 50 m reach, so uniform spacing puts ~80% of the gauges in the
    (trivially constant) flat region.  Instead, split roughly half the gauges
    onto the sloping bed, where d(x) actually varies.
    """
    n_slope = int(np.ceil(n_gauges / 2))
    n_flat  = n_gauges - n_slope
    flat    = np.linspace(GAUGE_X_MIN,  SLOPE_START, n_flat + 1)[:-1] if n_flat else np.empty(0)
    slope   = np.linspace(SLOPE_START, GAUGE_X_MAX, n_slope + 1)[1:]
    return np.concatenate([flat, slope])


# Observation generator using forward PINN
def generate_obs(n_gauges, noise_std, seed=0):
    """
    Generate synthetic observations from the TRAINED FORWARD PINN.

    Non-circular in the sense that the data source is a converged network rather
    than the linear closed form the inverse model would otherwise be fitting to
    itself.  It is still the same PDE family, so this tests identifiability of
    d(x), not model-form error.
    """
    rng     = np.random.default_rng(seed)
    gauge_x = gauge_positions(n_gauges)
    obs_t   = np.linspace(T_MIN, T_MAX, 80)
    obs_eta = np.zeros((n_gauges, len(obs_t)), dtype=np.float32)
    for i, xg in enumerate(gauge_x):
        x_g = np.full((len(obs_t),1), xg, dtype=np.float32)
        t_g = obs_t.reshape(-1,1).astype(np.float32)
        eta_pred,_,_ = fwd_sol(tf.constant(x_g), tf.constant(t_g))
        clean = eta_pred.numpy().flatten()
        obs_eta[i] = (clean + rng.normal(0., noise_std, clean.shape)).astype(np.float32)
    return gauge_x, obs_t, obs_eta, true_depth(gauge_x)


# Report gauge coverage of the slope for every K in the ablation
print(f'{"K":>4}  {"on slope":>9}  gauge depths [m]')
print('-'*72)
for K in GAUGE_COUNTS:
    gx  = gauge_positions(K)
    gd  = true_depth(gx)
    n_s = int(np.sum(gx > SLOPE_START))
    print(f'{K:>4}  {n_s:>4}/{K:<4}  ' + ' '.join(f'{v:.3f}' for v in gd))
    assert n_s >= 1, f'K={K}: no gauge on the slope'
    assert gd.ptp() > 0.1, f'K={K}: gauge depths span only {gd.ptp():.3f} m'
print('✓ every K samples a range of depths on the sloping bed')

# Quick visual: gauge signals from PINN vs linear approx
gx_test, ot_test, oe_test, gd_test = generate_obs(6, 0.001, seed=0)
omega_w = 2*np.pi/WAVE_PERIOD; k_lin = omega_w/np.sqrt(G*H0)

fig, axes = plt.subplots(1, 2, figsize=(13,4))
colors6 = plt.cm.tab10(np.linspace(0,1,6))
for i,(xg,col) in enumerate(zip(gx_test, colors6)):
    axes[0].plot(ot_test, oe_test[i], color=col, lw=0.8,
                 label=f'x={xg:.1f}, d={gd_test[i]:.2f}')
    lin = WAVE_AMP*np.cos(k_lin*(xg+30.5) - omega_w*ot_test)
    axes[1].plot(ot_test, lin,        color=col, lw=0.8)
axes[0].set(title='PINN-generated observations (non-circular)', xlabel='t [s]', ylabel=r'$\eta$ [m]')
axes[1].set(title='Linear approximation (old — circular)',      xlabel='t [s]', ylabel=r'$\eta$ [m]')
for ax in axes: ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'obs_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Inverse PINN — Prototype Run (K=6, σ=0.001 m)
Verify the inverse works before running the full ablation.

In [ ]:
K_PROTO, S_PROTO = 6, 0.001
print(f'Prototype inverse: K={K_PROTO}, σ={S_PROTO} m')

np.random.seed(0); tf.random.set_seed(0)
gx_p, ot_p, oe_p, od_p = generate_obs(K_PROTO, S_PROTO, seed=0)
coll_p = sample_collocation(seed=0)
obs_p  = build_obs(gx_p, ot_p, oe_p)
sol_p  = SolutionNet()
bm_p   = BathyNet()
_ = sol_p(coll_p['x_ic'], coll_p['t_ic'])

t0_p = time.time()
hist_p, t_a_p, t_l_p = train(sol_p, bm_p, coll_p, obs_p, verbose=True)
print(f'Done in {time.time()-t0_p:.1f}s')

# Results
d_pred_p  = bm_p.numpy(x_eval)
d_spl_p   = spline_baseline(gx_p, od_p, x_eval)
rmse_p, corr_p = bathy_metrics(d_true_eval, d_pred_p)
rmse_s, corr_s = bathy_metrics(d_true_eval, d_spl_p)
print(f'PINN   RMSE={rmse_p:.4f} m  Corr={corr_p:.4f}')
print(f'Spline RMSE={rmse_s:.4f} m  Corr={corr_s:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13,4))
axes[0].plot(x_eval, d_true_eval, 'k-',  lw=2.5, label='True d(x)')
axes[0].plot(x_eval, d_pred_p,    'r-',  lw=2,   label=f'PINN (RMSE={rmse_p:.4f})')
axes[0].plot(x_eval, d_spl_p,     'b--', lw=1.5, label=f'Spline (RMSE={rmse_s:.4f})')
axes[0].scatter(gx_p, od_p, color='green', zorder=5, s=60, label='Gauge depths')
axes[0].set(title=f'Prototype Recovery (K={K_PROTO}, σ={S_PROTO})',
            xlabel='x [m]', ylabel='d [m]')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].plot(x_eval, np.abs(d_true_eval-d_pred_p), 'r-',  lw=1.5, label='PINN |error|')
axes[1].plot(x_eval, np.abs(d_true_eval-d_spl_p),  'b--', lw=1.5, label='Spline |error|')
axes[1].scatter(gx_p, np.zeros_like(gx_p), color='green', zorder=5, s=40)
axes[1].set(title='Absolute Error', xlabel='x [m]', ylabel='|error| [m]')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'prototype_recovery.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Ensemble UQ — 5 Seeds
Train N_ENSEMBLE independent inverse PINNs with different random seeds.
Report mean ± 2σ band on recovered d(x) as uncertainty estimate.

In [ ]:
print(f'Ensemble UQ: {N_ENSEMBLE} seeds  (K={K_PROTO}, σ={S_PROTO})')
print('='*55)

ensemble_d, ensemble_rmse, ensemble_corr = [], [], []
timing_ensemble = []
hist_ensemble   = []

for seed in range(N_ENSEMBLE):
    np.random.seed(seed); tf.random.set_seed(seed)
    coll = sample_collocation(seed=seed)
    gx_e, ot_e, oe_e, _ = generate_obs(K_PROTO, S_PROTO, seed=seed)
    obs_e = build_obs(gx_e, ot_e, oe_e)
    sol_e = SolutionNet(); bm_e = BathyNet()
    _ = sol_e(coll['x_ic'], coll['t_ic'])
    t0 = time.time()
    hist_e, t_a, t_l = train(sol_e, bm_e, coll, obs_e, verbose=False)
    t_tot = time.time() - t0
    d_e = bm_e.numpy(x_eval)
    r, c = bathy_metrics(d_true_eval, d_e)
    ensemble_d.append(d_e)
    ensemble_rmse.append(r); ensemble_corr.append(c)
    hist_ensemble.append(hist_e)
    timing_ensemble.append({'seed':seed,'t_adam':t_a,'t_lbfgs':t_l,'t_total':t_tot})
    print(f'  Seed {seed}: RMSE={r:.4f} Corr={c:.4f} time={t_tot:.1f}s')

ensemble_d = np.array(ensemble_d)
d_mean = ensemble_d.mean(axis=0)
d_std  = ensemble_d.std(axis=0)
print(f'\nEnsemble RMSE: {np.mean(ensemble_rmse):.4f} ± {np.std(ensemble_rmse):.4f} m')
print(f'Ensemble Corr: {np.mean(ensemble_corr):.4f} ± {np.std(ensemble_corr):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,4))

# Left: recovery with uncertainty band
for d_i in ensemble_d:
    axes[0].plot(x_eval, d_i, alpha=0.2, lw=0.8, color='tomato')
axes[0].plot(x_eval, d_true_eval, 'k-', lw=2.5, label='True d(x)')
axes[0].plot(x_eval, d_mean, 'r-', lw=2,
             label=f'Mean (RMSE={np.mean(ensemble_rmse):.4f}±{np.std(ensemble_rmse):.4f})')
axes[0].fill_between(x_eval, d_mean-2*d_std, d_mean+2*d_std,
                     alpha=0.25, color='red', label='Mean ± 2σ')
axes[0].scatter(gx_p, od_p, color='green', zorder=5, s=50, label='Gauge depths')
axes[0].set(title=f'Ensemble UQ — {N_ENSEMBLE} seeds (K={K_PROTO}, σ={S_PROTO})',
            xlabel='x [m]', ylabel='d [m]')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# Right: uncertainty profile
axes[1].plot(x_eval, d_std, 'r-', lw=2)
axes[1].scatter(gx_p, np.zeros_like(gx_p), color='green', zorder=5, s=50, label='Gauges')
axes[1].set(title='Epistemic Uncertainty std(d(x))',
            xlabel='x [m]', ylabel='std [m]')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
axes[1].annotate(f'Max σ = {d_std.max():.4f} m\nat x = {x_eval[d_std.argmax()]:.1f} m',
                 xy=(x_eval[d_std.argmax()], d_std.max()),
                 xytext=(x_eval[d_std.argmax()]+5, d_std.max()*0.8),
                 fontsize=8, arrowprops=dict(arrowstyle='->'))

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'ensemble_uq.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Loss Weight Ablation (w_IC × w_data)
Systematically search the w_IC × w_data space to justify the chosen defaults.

In [ ]:
print(f'Weight ablation: {len(W_IC_GRID)}×{len(W_DATA_GRID)} = {len(W_IC_GRID)*len(W_DATA_GRID)} runs')
gx_w, ot_w, oe_w, _ = generate_obs(6, 0.001, seed=42)
obs_w = build_obs(gx_w, ot_w, oe_w)

weight_results = {}
for w_ic in W_IC_GRID:
    for w_data in W_DATA_GRID:
        np.random.seed(0); tf.random.set_seed(0)
        coll = sample_collocation(seed=0)
        sol  = SolutionNet(); bm = BathyNet()
        _ = sol(coll['x_ic'], coll['t_ic'])
        train(sol, bm, coll, obs_w, w_ic=w_ic, w_bc=W_BC, w_data=w_data, verbose=False)
        d_pred = bm.numpy(x_eval)
        r, c = bathy_metrics(d_true_eval, d_pred)
        weight_results[(w_ic,w_data)] = {'rmse':r,'corr':c}
        best = '★' if (w_ic==W_IC and w_data==W_DATA) else ' '
        print(f'  {best} w_IC={w_ic:>5}  w_data={w_data:>5}  RMSE={r:.4f}  Corr={c:.4f}')

print('✓ Done')

In [ ]:
rmse_mat = np.array([[weight_results[(wi,wd)]['rmse'] for wd in W_DATA_GRID] for wi in W_IC_GRID])
corr_mat = np.array([[weight_results[(wi,wd)]['corr'] for wd in W_DATA_GRID] for wi in W_IC_GRID])
best_idx = np.unravel_index(rmse_mat.argmin(), rmse_mat.shape)

fig, axes = plt.subplots(1, 2, figsize=(11,4))
for ax, mat, title, cmap in zip(axes, [rmse_mat, corr_mat],
                                 ['RMSE [m]','Correlation'], ['YlOrRd','YlGn']):
    im = ax.imshow(mat, cmap=cmap, aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(W_DATA_GRID))); ax.set_xticklabels([str(w) for w in W_DATA_GRID])
    ax.set_yticks(range(len(W_IC_GRID)));   ax.set_yticklabels([str(w) for w in W_IC_GRID])
    ax.set_xlabel('w_data'); ax.set_ylabel('w_IC'); ax.set_title(title)
    for i in range(len(W_IC_GRID)):
        for j in range(len(W_DATA_GRID)):
            ax.text(j, i, f'{mat[i,j]:.3f}', ha='center', va='center', fontsize=9)

# Mark best and default
axes[0].add_patch(plt.Rectangle((best_idx[1]-0.5, best_idx[0]-0.5), 1, 1,
                                  fill=False, edgecolor='blue', lw=2.5, label='Best'))
def_i = W_IC_GRID.index(W_IC); def_j = W_DATA_GRID.index(W_DATA)
axes[0].add_patch(plt.Rectangle((def_j-0.5, def_i-0.5), 1, 1,
                                  fill=False, edgecolor='green', lw=2, ls='--', label='Default'))
axes[0].legend(fontsize=8)

plt.suptitle('Loss Weight Ablation (K=6, σ=0.001 m)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'weight_ablation.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Best: w_IC={W_IC_GRID[best_idx[0]]}, w_data={W_DATA_GRID[best_idx[1]]} → RMSE={rmse_mat[best_idx]:.4f}')
print(f'Default: w_IC={W_IC}, w_data={W_DATA} → RMSE={rmse_mat[def_i,def_j]:.4f}')

## 10. Convergence Analysis Across Seeds
Overlay loss curves for all ensemble seeds to show reproducibility.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,4))
colors_s = plt.cm.tab10(np.linspace(0, 1, N_ENSEMBLE))

for hist_e, col, seed in zip(hist_ensemble, colors_s, range(N_ENSEMBLE)):
    ep = np.arange(1, len(hist_e['total'])+1)
    axes[0].semilogy(ep, hist_e['total'], color=col, lw=1.2, alpha=0.8, label=f'seed {seed}')
    axes[1].semilogy(ep, hist_e['data'],  color=col, lw=1.2, alpha=0.8)

# Mean ± std band
min_len   = min(len(h['total']) for h in hist_ensemble)
total_arr = np.array([h['total'][:min_len] for h in hist_ensemble])
data_arr  = np.array([h['data'][:min_len]  for h in hist_ensemble])
ep_c      = np.arange(1, min_len+1)

for ax, arr, title in zip(axes, [total_arr, data_arr], ['Total Loss','Data Loss']):
    ax.semilogy(ep_c, arr.mean(axis=0), 'k--', lw=2, label='Mean', zorder=5)
    ax.axvline(EPOCHS_ADAM, color='gray', ls=':', lw=1, label='Adam→LBFGS')
    ax.set(title=title, xlabel='Epoch', ylabel='Loss')
    ax.legend(fontsize=8, ncol=3); ax.grid(True, which='both', alpha=0.3)

plt.suptitle(f'Convergence Across {N_ENSEMBLE} Seeds (K={K_PROTO}, σ={S_PROTO})')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'convergence_seeds.png'), dpi=150, bbox_inches='tight')
plt.show()

final = [h['total'][-1] for h in hist_ensemble]
print(f'Final total loss: {np.mean(final):.3e} ± {np.std(final):.3e}')

## 11. Full Ablation — K × σ (PINN-generated observations)
Main results table for the paper. Uses non-circular observations from the forward PINN.

In [ ]:
results = {}
n_total = len(GAUGE_COUNTS) * len(NOISE_LEVELS)

for idx, (K, sigma) in enumerate([(k,s) for k in GAUGE_COUNTS for s in NOISE_LEVELS], 1):
    print(f'[{idx}/{n_total}] K={K}, σ={sigma}')
    np.random.seed(idx); tf.random.set_seed(idx)
    gx, ot, oe, od = generate_obs(K, sigma, seed=idx)
    obs  = build_obs(gx, ot, oe)
    coll = sample_collocation(seed=0)
    sol  = SolutionNet(); bm = BathyNet()
    _ = sol(coll['x_ic'], coll['t_ic'])
    train(sol, bm, coll, obs, verbose=False)
    d_pred = bm.numpy(x_eval)
    d_spl  = spline_baseline(gx, od, x_eval)
    rp, cp = bathy_metrics(d_true_eval, d_pred)
    rs, cs = bathy_metrics(d_true_eval, d_spl)
    results[(K,sigma)] = dict(rmse_pinn=rp,corr_pinn=cp,rmse_spline=rs,corr_spline=cs,
                               d_pred=d_pred,d_spline=d_spl,gauge_x=gx,obs_depth=od)
    print(f'  PINN RMSE={rp:.4f} Corr={cp:.4f} | Spline RMSE={rs:.4f} Corr={cs:.4f}')

print('\n✓ Full ablation complete')

In [ ]:
# Table
hdr = f"{'K':>4}  {'σ':>7}  {'PINN RMSE':>11}  {'PINN Corr':>10}  {'Spl RMSE':>10}  {'Spl Corr':>9}"
print(hdr); print('-'*60)
rows = []
for K in GAUGE_COUNTS:
    for sigma in NOISE_LEVELS:
        r = results[(K,sigma)]
        row = (f"{K:>4}  {sigma:>7.3f}  {r['rmse_pinn']:>11.4f}  {r['corr_pinn']:>10.4f}  "
               f"{r['rmse_spline']:>10.4f}  {r['corr_spline']:>9.4f}")
        print(row); rows.append(row)

# encoding='utf-8' is required: the header contains σ, which the Windows
# default (cp1252) cannot encode.
with open(os.path.join(SAVE_DIR,'ablation_table.txt'),'w', encoding='utf-8') as f:
    f.write(hdr+'\n'+'-'*60+'\n'+'\n'.join(rows)+'\n')
print(f"\nSaved → {os.path.join(SAVE_DIR,'ablation_table.txt')}")

# Heatmap
mat_pinn = np.array([[results[(K,s)]['rmse_pinn']   for K in GAUGE_COUNTS] for s in NOISE_LEVELS])
mat_spl  = np.array([[results[(K,s)]['rmse_spline'] for K in GAUGE_COUNTS] for s in NOISE_LEVELS])
fig, axes = plt.subplots(1, 2, figsize=(11,4))
for ax, mat, title in zip(axes,[mat_pinn,mat_spl],['PINN RMSE [m]','Spline RMSE [m]']):
    im = ax.imshow(mat, cmap='YlOrRd', aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(GAUGE_COUNTS))); ax.set_xticklabels([f'K={k}' for k in GAUGE_COUNTS])
    ax.set_yticks(range(len(NOISE_LEVELS))); ax.set_yticklabels([f'σ={s}' for s in NOISE_LEVELS])
    ax.set_xlabel('Gauge count K'); ax.set_ylabel('Noise σ [m]'); ax.set_title(title)
    for i in range(len(NOISE_LEVELS)):
        for j in range(len(GAUGE_COUNTS)):
            ax.text(j,i,f'{mat[i,j]:.3f}',ha='center',va='center',fontsize=9)
plt.suptitle('Bathymetry Recovery RMSE — PINN vs Spline (PINN-generated obs)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'rmse_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

wins = sum(1 for r in results.values() if r['rmse_pinn'] < r['rmse_spline'])
print(f'PINN beats spline in {wins}/{len(results)} configurations')
print('NOTE: the spline is given the TRUE depth at each gauge; the PINN sees only η. '
      'Report this asymmetry alongside the comparison.')

## 12. κ₁ Sensitivity Analysis
Train a forward PINN for each κ₁d value. RMSE computed relative to reference (κ₁d=1.52).

In [ ]:
print('κ₁ sensitivity analysis...')

# Reference already trained (fwd_sol at κ₁d = KAPPA1_D)
NX_K, NT_K = 200, 50
x_kg = np.linspace(X_MIN, X_MAX, NX_K, dtype=np.float32)
t_kg = np.linspace(T_MIN, T_MAX, NT_K, dtype=np.float32)
XXk, TTk = np.meshgrid(x_kg, t_kg)
eta_ref_k,_,_ = fwd_sol(
    tf.constant(XXk.flatten()[:,None]),
    tf.constant(TTk.flatten()[:,None])
)
ETA_ref = eta_ref_k.numpy().reshape(NT_K, NX_K)

kappa_results = {}
coll_k = sample_collocation(seed=0)   # same collocation for all

for k1d in KAPPA1_D_VALUES:
    a, b, g = vbm_coeffs(k1d, H0)     # reported at d₀; the PDE uses local depth
    np.random.seed(0); tf.random.set_seed(0)
    sol_k = SolutionNet()
    _ = sol_k(coll_k['x_ic'], coll_k['t_ic'])
    hist_k, t_ak, t_lk = train(
        sol_k, true_depth_tf, coll_k, obs=None, kappa1_d=k1d, verbose=False
    )
    eta_k,_,_ = sol_k(tf.constant(XXk.flatten()[:,None]),
                       tf.constant(TTk.flatten()[:,None]))
    ETA_k = eta_k.numpy().reshape(NT_K, NX_K)
    rmse_ref = np.sqrt(np.mean((ETA_k - ETA_ref)**2))
    kappa_results[k1d] = {
        'alpha':a,'beta':b,'gamma':g,'ETA':ETA_k,
        'rmse_vs_ref':rmse_ref,'final_loss':hist_k['total'][-1],
        't_adam':t_ak,'t_lbfgs':t_lk
    }
    marker = ' ← optimal (= reference)' if k1d == KAPPA1_D else ''
    print(f'  κ₁d={k1d:.2f}: RMSE vs ref={rmse_ref:.4f}  loss={hist_k["total"][-1]:.3e}{marker}')

# The κ₁d = KAPPA1_D run reuses the reference's seed, collocation and coefficients,
# so it must reproduce fwd_sol to near machine precision.  If it does not, the
# reference and the sweep have drifted apart.
_r_opt = kappa_results[KAPPA1_D]['rmse_vs_ref']
print(f'\nSelf-consistency at κ₁d={KAPPA1_D}: RMSE vs reference = {_r_opt:.2e}')
if _r_opt > 1e-3:
    print('  ⚠ expected ≈0 — reference and sweep are not using identical settings')
print('✓ Done')

In [ ]:
k1d_arr  = list(kappa_results.keys())
rmse_arr = [kappa_results[k]['rmse_vs_ref'] for k in k1d_arr]

fig, axes = plt.subplots(1, 2, figsize=(13,4))

axes[0].plot(k1d_arr, rmse_arr, 'o-', color='steelblue', lw=2, ms=8)
axes[0].axvline(KAPPA1_D, color='red', ls='--', lw=1.5, label=f'Optimal κ₁d={KAPPA1_D}')
for k1d, r in zip(k1d_arr, rmse_arr):
    axes[0].annotate(f'{r:.4f}', (k1d, r), textcoords='offset points',
                     xytext=(0,8), ha='center', fontsize=8)
axes[0].set(title=r'RMSE vs Reference as Function of $\kappa_1 d$',
            xlabel=r'$\kappa_1 d$', ylabel='RMSE [m]')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

t_snap = NT_K // 2
colors_k = plt.cm.plasma(np.linspace(0.1, 0.9, len(k1d_arr)))
axes[1].plot(x_kg, ETA_ref[t_snap], 'k-', lw=2.5, label=f'Reference (κ₁d={KAPPA1_D})')
for k1d, col in zip(k1d_arr, colors_k):
    if k1d != KAPPA1_D:
        axes[1].plot(x_kg, kappa_results[k1d]['ETA'][t_snap],
                     '-', color=col, lw=1.2, alpha=0.8, label=f'κ₁d={k1d}')
axes[1].set(title=f'η at t={t_kg[t_snap]:.1f}s for different κ₁d',
            xlabel='x [m]', ylabel=r'$\eta$ [m]')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'kappa_sensitivity.png'), dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f'{"κ₁d":>6}  {"α(d₀)":>8}  {"β(d₀)":>8}  {"RMSE vs ref":>13}  {"Final loss":>12}')
print('-'*56)
kappa_rows = []
for k1d in k1d_arr:
    r = kappa_results[k1d]
    marker = ' *' if k1d == KAPPA1_D else ''
    row = f"{k1d:>6.2f}  {r['alpha']:>8.4f}  {r['beta']:>8.4f}  {r['rmse_vs_ref']:>13.4f}  {r['final_loss']:>12.3e}{marker}"
    print(row); kappa_rows.append(row)
with open(os.path.join(SAVE_DIR,'kappa_sensitivity.txt'),'w', encoding='utf-8') as f:
    f.write(f'kappa_1 Sensitivity (* = optimal = {KAPPA1_D})\n'+'\n'.join(kappa_rows)+'\n')

## 13. PDE Residual Field Visualization
Shows *where* spatially and temporally the PINN satisfies/violates each equation.

In [ ]:
NX_R, NT_R = 150, 80
x_r = np.linspace(X_MIN, X_MAX, NX_R, dtype=np.float32)
t_r = np.linspace(T_MIN, T_MAX, NT_R, dtype=np.float32)
XXr, TTr = np.meshgrid(x_r, t_r)
x_rt = tf.constant(XXr.flatten()[:,None].astype(np.float32))
t_rt = tf.constant(TTr.flatten()[:,None].astype(np.float32))

# Reuse the training residual so the diagnostic cannot drift from the physics
# that was actually optimised (the old cell recomputed them by hand with
# h = H0 + eta and constant alpha/beta — a different PDE).
r13, r14, r15 = compute_residuals(fwd_sol, true_depth_tf, x_rt, t_rt, KAPPA1_D)
R13 = r13.numpy().reshape(NT_R, NX_R)
R14 = r14.numpy().reshape(NT_R, NX_R)
R15 = r15.numpy().reshape(NT_R, NX_R)

fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, R, title in zip(axes, [R13, R14, R15],
    [r'Residual Eq.(13): $\partial_t\eta+\partial_x(hu)+\partial_x(\beta\partial_x\Psi)$',
     r'Residual Eq.(14): $\partial_t u+g\partial_x\eta+u\partial_x u$',
     r'Residual Eq.(15): $-\partial_x(\alpha\partial_x\Psi)+\gamma\Psi-\partial_x(\beta u)$']):
    vmax = np.abs(R).max()
    im = ax.pcolormesh(x_r, t_r, R, cmap='RdBu_r', vmin=-vmax, vmax=vmax, shading='auto')
    plt.colorbar(im, ax=ax)
    ax.axvline(SLOPE_START, color='k', ls=':', lw=1)
    ax.axvline(X_SHORE,     color='k', ls='--', lw=1)
    ax.set(title=title, xlabel='x [m]', ylabel='t [s]')
axes[0].text(SLOPE_START, T_MAX*0.97, ' toe', fontsize=7, va='top')
axes[0].text(X_SHORE,     T_MAX*0.97, ' shore', fontsize=7, va='top')

plt.suptitle(f'PDE Residual Fields — Forward PINN (κ₁d={KAPPA1_D})', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'residual_fields.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean |residual| Eq.13: {np.abs(R13).mean():.3e}')
print(f'Mean |residual| Eq.14: {np.abs(R14).mean():.3e}')
print(f'Mean |residual| Eq.15: {np.abs(R15).mean():.3e}')

# Split by region — the dry beach is where the thin-film regularisation acts
wet = x_r < X_SHORE
print(f'\nBy region (Eq.13):  offshore/slope  {np.abs(R13[:, wet]).mean():.3e}'
      f'   dry beach  {np.abs(R13[:, ~wet]).mean():.3e}')

## 14. Computational Cost Table

In [ ]:
# Inference timing
x_inf = XXr.flatten()[:,None].astype(np.float32)
t_inf = TTr.flatten()[:,None].astype(np.float32)
t_inf_start = time.time()
for _ in range(10):
    _ = fwd_sol(tf.constant(x_inf), tf.constant(t_inf))
t_inf_ms = (time.time()-t_inf_start)/10*1000

# Ensemble mean timing
t_adam_mean  = np.mean([t['t_adam']  for t in timing_ensemble])
t_adam_std   = np.std([t['t_adam']   for t in timing_ensemble])
t_total_mean = np.mean([t['t_total'] for t in timing_ensemble])
t_total_std  = np.std([t['t_total']  for t in timing_ensemble])
t_lbfgs_mean = np.mean([t['t_lbfgs'] for t in timing_ensemble])

# κ₁ sensitivity timings
kappa_timing = {k: kappa_results[k]['t_adam']+kappa_results[k]['t_lbfgs']
                for k in k1d_arr}

hw = 'GPU' if len(tf.config.list_physical_devices('GPU'))>0 else 'CPU'
print('='*65)
print(f'COMPUTATIONAL COST  ({hw})')
print('='*65)
print(f'Forward PINN (κ₁d={KAPPA1_D}):')
print(f'  Adam ({EPOCHS_ADAM} ep)  : {t_adam_fwd:.1f}s')
print(f'  L-BFGS ({EPOCHS_LBFGS} it): {t_lbfgs_fwd:.1f}s')
print(f'  Total             : {t_fwd_total:.1f}s')
print(f'Inverse PINN (mean±std, n={N_ENSEMBLE} seeds):')
print(f'  Adam   : {t_adam_mean:.1f} ± {t_adam_std:.1f}s')
print(f'  L-BFGS : {t_lbfgs_mean:.1f}s (mean)')
print(f'  Total  : {t_total_mean:.1f} ± {t_total_std:.1f}s')
print(f'Inference ({NX_R*NT_R} pts): {t_inf_ms:.2f} ms')
print(f'Network params: {fwd_sol.count_params():,}')
print('='*65)

# Save
timing_lines = [
    f'Hardware: {hw}',
    f'kappa1_d: {KAPPA1_D}',
    f'Forward PINN: Adam={t_adam_fwd:.1f}s LBFGS={t_lbfgs_fwd:.1f}s Total={t_fwd_total:.1f}s',
    f'Inverse PINN (mean+-std/{N_ENSEMBLE} seeds): Adam={t_adam_mean:.1f}+-{t_adam_std:.1f}s Total={t_total_mean:.1f}+-{t_total_std:.1f}s',
    f'Inference ({NX_R*NT_R} pts): {t_inf_ms:.2f}ms',
]
with open(os.path.join(SAVE_DIR,'timing.txt'),'w', encoding='utf-8') as f:
    f.write('\n'.join(timing_lines)+'\n')

# LaTeX snippet
latex = [
    '% --- Paste into paper ---',
    '\\begin{table}[h]\\centering',
    '\\caption{Computational cost summary.}',
    '\\begin{tabular}{lccc}\\toprule',
    'Task & Adam [s] & L-BFGS [s] & Total [s] \\\\\\midrule',
    f'Forward PINN & {t_adam_fwd:.1f} & {t_lbfgs_fwd:.1f} & {t_fwd_total:.1f} \\\\',
    f'Inverse PINN (mean$\\pm$std) & {t_adam_mean:.1f}$\\pm${t_adam_std:.1f} & {t_lbfgs_mean:.1f} & {t_total_mean:.1f}$\\pm${t_total_std:.1f} \\\\',
    f'\\multicolumn{{4}}{{l}}{{Inference ({NX_R*NT_R} pts): {t_inf_ms:.2f}~ms; Hardware: {hw}}} \\\\',
    '\\bottomrule\\end{tabular}\\end{table}',
]
with open(os.path.join(SAVE_DIR,'timing_latex.tex'),'w', encoding='utf-8') as f:
    f.write('\n'.join(latex)+'\n')
print(f"LaTeX snippet → {os.path.join(SAVE_DIR,'timing_latex.tex')}")

## 15. Summary of All Outputs

In [ ]:
print(f'All outputs saved to: {SAVE_DIR}/')
print()
for f in sorted(os.listdir(SAVE_DIR)):
    sz = os.path.getsize(os.path.join(SAVE_DIR,f))/1024
    print(f'  {f:<42} {sz:>6.1f} KB')

print()
print('KEY RESULTS')
print('='*55)
best_cfg = min(results, key=lambda k: results[k]['rmse_pinn'])
print(f'Best inverse config: K={best_cfg[0]}, σ={best_cfg[1]}')
print(f'  RMSE={results[best_cfg]["rmse_pinn"]:.4f}  Corr={results[best_cfg]["corr_pinn"]:.4f}')
print(f'Ensemble UQ: RMSE={np.mean(ensemble_rmse):.4f}±{np.std(ensemble_rmse):.4f}')
best_w = min(weight_results, key=lambda k: weight_results[k]['rmse'])
print(f'Best weights: w_IC={best_w[0]}, w_data={best_w[1]} → RMSE={weight_results[best_w]["rmse"]:.4f}')
worst_k = max(k1d_arr, key=lambda k: kappa_results[k]['rmse_vs_ref'])
print(f'κ₁ sensitivity: worst at κ₁d={worst_k} (RMSE={kappa_results[worst_k]["rmse_vs_ref"]:.4f})')
print(f'PINN beats spline in {wins}/{len(results)} ablation configs')
print('='*55)